In [1]:
import json
import time
import random
import re
from tqdm.auto import tqdm
from deep_translator import GoogleTranslator

paraphrase_map = {
    "what is this?": ["Cái gì đây?", "Đây là vật gì?", "Món đồ này là gì?", "Bạn có thể cho tôi biết đây là cái gì không?", "Trong hình là vật gì vậy?"],
    "what color is this?": ["Món đồ này màu gì?", "Màu sắc của vật này là gì?", "Vật này có màu gì vậy?", "Bạn nhìn giúp tôi xem cái này màu gì nhé."],
    "what is it?": ["Nó là gì vậy?", "Đây là cái gì thế?", "Vật thể này là gì?", "Cái này gọi là gì?"],
    "what's this?": ["Cái này là gì?", "Cái gì thế này?", "Đây là món gì?", "Cho tôi hỏi đây là vật gì?"],
    "what is in this box?": ["Trong hộp này có gì?", "Có gì bên trong chiếc hộp này vậy?", "Bên trong hộp chứa cái gì?", "Cái hộp này đựng gì thế?"],
    "what is this item?": ["Món đồ này là gì?", "Đây là vật dụng gì?", "Vật này được gọi là gì?", "Tên của món đồ này là gì?"],
    "what does this say?": ["Chữ này viết gì vậy?", "Trên này ghi nội dung gì?", "Dòng chữ này có nghĩa là gì?", "Bạn đọc giúp tôi chữ trên này với."],
    "what color is this shirt?": ["Cái áo này màu gì?", "Áo này có màu sắc ra sao?", "Màu của chiếc áo này là gì?", "Chiếc áo trong hình màu gì vậy?"],
    "what flavor is this?": ["Đây là vị gì?", "Hương vị của món này là gì?", "Cái này có mùi vị thế nào?", "Món này vị gì vậy?"],
    "what is in this can?": ["Trong lon này có gì?", "Có gì bên trong chiếc lon này vậy?", "Cái lon này chứa gì thế?", "Bạn xem giúp tôi trong lon này đựng gì với."]
}




/home/check/Programs/anaconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def translate_json_questions(input_file, output_file):
    print(f"Đang đọc dữ liệu từ: {input_file}")
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    unique_questions = list(set([item['question'] for item in data if 'question' in item]))
    print(f"Tìm thấy {len(unique_questions)} câu hỏi duy nhất cần xử lý.")

    translated_dict = {}
    needs_translation = []
    
    for q in unique_questions:
        q_clean = str(q).strip().lower().rstrip('?') + "?"
        if q_clean in paraphrase_map:
            translated_dict[q] = random.choice(paraphrase_map[q_clean])
        else:
            needs_translation.append(q)

    # 4. DỊCH SIÊU TỐC THEO BATCH (Cho những câu chưa có trong từ điển)
    if needs_translation:
        print(f"Đang gửi {len(needs_translation)} câu cho Google Translate...")
        translator = GoogleTranslator(source='en', target='vi')
        batch_size = 50 
        
        for i in tqdm(range(0, len(needs_translation), batch_size), desc="Đang dịch Question"):
            chunk = needs_translation[i : i + batch_size]
            
            # Gộp câu để gửi API 1 lần
            joined_text = "\n".join([str(t).replace('\n', ' ').strip() for t in chunk])
            
            try:
                translated_text = translator.translate(joined_text)
                vi_texts = [t.strip() for t in translated_text.split('\n')]
                
                # Check xem số lượng kết quả trả về có khớp số lượng gửi đi không
                if len(vi_texts) == len(chunk):
                    for j, orig_text in enumerate(chunk):
                        translated_dict[orig_text] = vi_texts[j]
                else:
                    # Nếu Google nối dòng sai, tiến hành dịch lẻ từng câu để fallback
                    for orig_text in chunk:
                        translated_dict[orig_text] = translator.translate(str(orig_text))
                        time.sleep(0.1)
                        
                time.sleep(0.5) # Nghỉ để tránh Rate Limit
                
            except Exception as e:
                print(f"\n[!] Lỗi ở batch từ {i} đến {i+batch_size}: {e}")
                print("[!] Tạm nghỉ 5 giây...")
                time.sleep(5)
    # 5. ĐẮP BẢN DỊCH VÀO LẠI FILE JSON
    print("Đang cập nhật lại file JSON...")
    for item in data:
        if 'question' in item:
            orig_q = item['question']
            
            # Tùy chọn 1: Ghi đè thẳng câu tiếng Việt vào 'question'
            item['question'] = translated_dict.get(orig_q, orig_q)
            
            # Tùy chọn 2 (Nếu bạn muốn giữ câu tiếng Anh gốc thì dùng dòng dưới đây thay cho dòng trên):
            # item['question_vi'] = translated_dict.get(orig_q, orig_q)

    # 6. LƯU FILE KẾT QUẢ
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
        
    print(f"✅ Hoàn tất! Đã lưu kết quả tại: {output_file}")


In [5]:
if __name__ == "__main__":
    # Thay đổi tên file theo file thực tế của bạn
    INPUT_1_JSON = "vizwiz_dataset_1000/pixtral_ans_1.json"
    OUTPUT_1_JSON = "pixtral_ans_1_translated.json"

    INPUT_0_JSON = "vizwiz_dataset_1000/pixtral_ans_0.json"
    OUTPUT_0_JSON = "pixtral_ans_0_translated.json"

    translate_json_questions(INPUT_1_JSON, OUTPUT_1_JSON)
    translate_json_questions(INPUT_0_JSON, OUTPUT_0_JSON)

Đang đọc dữ liệu từ: vizwiz_dataset_1000/pixtral_ans_1.json
Tìm thấy 405 câu hỏi duy nhất cần xử lý.
Đang gửi 385 câu cho Google Translate...


Đang dịch Question: 100%|██████████| 8/8 [00:10<00:00,  1.29s/it]


Đang cập nhật lại file JSON...
✅ Hoàn tất! Đã lưu kết quả tại: pixtral_ans_1_translated.json
Đang đọc dữ liệu từ: vizwiz_dataset_1000/pixtral_ans_0.json
Tìm thấy 332 câu hỏi duy nhất cần xử lý.
Đang gửi 321 câu cho Google Translate...


Đang dịch Question: 100%|██████████| 7/7 [00:10<00:00,  1.54s/it]

Đang cập nhật lại file JSON...
✅ Hoàn tất! Đã lưu kết quả tại: pixtral_ans_0_translated.json
